In [ ]:
# =====================================================================
# INSTALACIÓN PREVIA 
# # !pip install openpyxl
# =====================================================================
!pip install openpyxl

In [ ]:
import numpy as np
import pandas as pd

🚀 Iniciando Simulador de Modelos de Distribución Urbana para: City ​​of Barcelona
📍 Analizando sector geográfico del barrio: EIXAMPLE

📊 --- RESULTADOS COMPARATIVOS PARA EL BARRIO: EIXAMPLE ---
• Centro Logístico de Origen Real: DCT9 Amazon Logistics
• Distancia desde el CC al sector urbano: 5.73 km
• Total de paquetes a distribuir: 416

                                    Km Recorridos  Número Viajes  Emissions CO2 (kg)  Costo Total (€)
M1: Furgoneta Combustión desde CC          123.34            7.0               27.14           156.42
M2: Furgoneta Eléctrica desde CC           123.34            7.0                0.00           135.68
M3: CC -> Microhub -> Bicicleta             61.10           22.0                2.52           102.28
M4: CC -> PUDO -> Entrega a pie            881.97           36.0                2.52          2534.54
M5: CC -> PUDO -> Recogida Cliente         881.97          417.0               24.28           213.15


In [ ]:

# =====================================================================
# 1. CONFIGURACIÓN DEL ÁMBITO DE ESTUDIO
# =====================================================================
# Nombres de los archivos .xlsx subidos a la carpeta raíz de Colab
archivo_puntos_b2c = "Points B2C_20250402.xlsx"
archivo_centros_cc = "CC.xlsx"

In [ ]:

# --- CONFIGURA AQUÍ TU ESCENARIO ---
# Pestañas válidas en el Excel: 'City ​​of Barcelona', 'City ​​of Madrid', 'City ​​of Valencia'
ciudad_activa = "City ​​of Barcelona"

# Barrios configurados para el estudio:
# - Con Barcelona: 'Ciutat Vella', 'Eixample', 'El Carmel'
# - Con Madrid:    'Lavapiés', 'Moratalaz', 'El Pardo'
# - Con Valencia:  'Benicalap', 'Camins al Grau', 'La Punta'
barrio_activo = "Eixample"

print(f"🚀 Iniciando Simulador de Modelos de Distribución Urbana para: {ciudad_activa}")
print(f"📍 Analizando sector geográfico del barrio: {barrio_activo.upper()}\n")

In [ ]:
# =====================================================================
# 2. LECTURA OPERATIVA DE EXCEL DESDE EL ENTORNO VIRTUAL
# =====================================================================
# 'header=1' salta la primera fila de cortesía/decorativa del Excel de origen
df_puntos_raw = pd.read_excel(archivo_puntos_b2c, sheet_name=ciudad_activa, header=1)
df_centros_raw = pd.read_excel(archivo_centros_cc, sheet_name=ciudad_activa, header=1)

In [ ]:

# Homologar formato de coordenadas (corrige problemas si vienen mapeadas con comas decimales)
for df in [df_puntos_raw, df_centros_raw]:
    df["Latitude"] = pd.to_numeric(
        df["Latitude"].astype(str).str.replace(",", "."), errors="coerce"
    )
    df["Longitude"] = pd.to_numeric(
        df["Longitude"].astype(str).str.replace(",", "."), errors="coerce"
    )

df_puntos = df_puntos_raw.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)
df_centros = df_centros_raw.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

In [ ]:
# =====================================================================
# 3. FILTRADO Y DELIMITACIÓN GEOGRÁFICA DE LOS 9 BARRIOS DE ESTUDIO
# =====================================================================
# Cajas de coordenadas (Bounding Boxes) aproximadas para segmentar la demanda de cada barriada
limites_barrios = {
    # Barcelona
    "Eixample": {"lat": (41.380, 41.405), "lon": (2.145, 2.175)},
    "Ciutat Vella": {"lat": (41.370, 41.390), "lon": (2.160, 2.190)},
    "El Carmel": {"lat": (41.415, 41.435), "lon": (2.145, 2.165)},
    # Madrid
    "Lavapiés": {"lat": (40.405, 40.415), "lon": (-3.708, -3.693)},
    "Moratalaz": {"lat": (40.400, 40.420), "lon": (-3.660, -3.630)},
    "El Pardo": {"lat": (40.500, 40.550), "lon": (-3.800, -3.750)},
    # Valencia
    "Benicalap": {"lat": (39.485, 39.505), "lon": (-0.400, -0.380)},
    "Camins al Grau": {"lat": (39.455, 39.475), "lon": (-0.360, -0.330)},
    "La Punta": {"lat": (39.430, 39.455), "lon": (-0.350, -0.320)},
}

if barrio_activo in limites_barrios:
    limite = limites_barrios[barrio_activo]
    df_destinos_barrio = df_puntos[
        (df_puntos["Latitude"].between(limite["lat"][0], limite["lat"][1]))
        & (df_puntos["Longitude"].between(limite["lon"][0], limite["lon"][1]))
    ].copy()
else:
    print(
        f"⚠️ El barrio '{barrio_activo}' no está parametrizado. Se procesará la pestaña completa."
    )
    df_destinos_barrio = df_puntos.copy()

num_paquetes = len(df_destinos_barrio)
if num_paquetes == 0:
    print(
        f"❌ Error: No se detectan puntos B2C dentro del cuadrante geográfico de {barrio_activo}."
    )
    print(
        "Prueba ampliando u homologando los límites de latitud/longitud en el diccionario."
    )
else:
    # Centroide del barrio para calcular la localización predictiva de Microhubs y PUDOs
    barrio_lat_centro = df_destinos_barrio["Latitude"].mean()
    barrio_lon_centro = df_destinos_barrio["Longitude"].mean()

    # =====================================================================
    # 4. MATRIZ DE PARÁMETROS OPERATIVOS DE LOS 5 MODELOS URBANOS
    # =====================================================================
    PARAMETROS = {
        "FURGONETA_CONV": {
            "costo_km": 0.45,  # Combustible + mantenimiento (€/km)
            "costo_hora": 18.0,  # Salario del conductor (€/h)
            "v_media": 22.0,  # Velocidad media urbana (km/h)
            "co2_km": 220.0,  # gramos de CO2 por km recorrido
            "capacidad": 60,  # Capacidad máxima de paquetes por viaje
        },
        "FURGONETA_ELEC": {
            "costo_km": 0.20,
            "costo_hora": 18.0,
            "v_media": 20.0,
            "co2_km": 0.0,  # Emisiones directas cero
            "capacidad": 60,
        },
        "BICICLETA_CARGO": {
            "costo_km": 0.05,
            "costo_hora": 14.0,
            "v_media": 14.0,
            "co2_km": 0.0,
            "capacidad": 20,
            "fijo_hub_dia": 45.0,  # Costo diario de estructura del microhub
        },
        "PUDO_A_PIE": {
            "costo_km": 0.0,
            "costo_hora": 12.0,
            "v_media": 4.5,
            "co2_km": 0.0,
            "capacidad": 12,
            "comision_pudo": 0.50,  # Costo fijo por paquete gestionado en tienda
        },
        "PUDO_CONSUMIDOR": {
            "comision_pudo": 0.50,
            "co2_km_estimado_cliente": 25.0,  # Media considerando que algunos van en coche
        },
    }

    # =====================================================================
    # 5. FUNCIONES PARA SIMULAR RUTAS (Algoritmo Nearest Neighbor para TSP)
    # =====================================================================
    def calcular_haversine(lat1, lon1, lat2, lon2):
        R = 6371.0  # Radio medio de la Tierra en km
        l1, o1, l2, o2 = (
            np.radians(lat1),
            np.radians(lon1),
            np.radians(lat2),
            np.radians(lon2),
        )
        dlat, dlon = l2 - l1, o2 - o1
        a = (
            np.sin(dlat / 2) ** 2
            + np.cos(l1) * np.cos(l2) * np.sin(dlon / 2) ** 2
        )
        return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))

    def simular_kilometros_tsp(inicio_lat, inicio_lon, df_puntos_ruta):
        puntos_restantes = df_puntos_ruta.copy()
        km_acumulados = 0.0
        pos_actual = (inicio_lat, inicio_lon)

        while len(puntos_restantes) > 0:
            distancias = puntos_restantes.apply(
                lambda r: calcular_haversine(
                    pos_actual[0], pos_actual[1], r["Latitude"], r["Longitude"]
                ),
                axis=1,
            )
            idx_cercano = distancias.idxmin()
            km_acumulados += distancias.min()
            pos_actual = (
                puntos_restantes.loc[idx_cercano, "Latitude"],
                puntos_restantes.loc[idx_cercano, "Longitude"],
            )
            puntos_restantes = puntos_restantes.drop(idx_cercano)

        km_acumulados += calcular_haversine(
            pos_actual[0], pos_actual[1], inicio_lat, inicio_lon
        )
        return km_acumulados

    # =====================================================================
    # 6. SIMULACIÓN MULTICRITERIO DE LOS 5 MODELOS LOGÍSTICOS
    # =====================================================================
    # Encontrar qué Almacén CC real de la pestaña queda más cerca del barrio (Asignación)
    df_centros["dist_al_barrio"] = df_centros.apply(
        lambda c: calcular_haversine(
            barrio_lat_centro, barrio_lon_centro, c["Latitude"], c["Longitude"]
        ),
        axis=1,
    )
    cc_seleccionado = df_centros.loc[df_centros["dist_al_barrio"].idxmin()]
    distancia_troncal = cc_seleccionado["dist_al_barrio"]

    informe_final = {}

    # --- MODELO 1: Furgoneta de Combustión desde CC ---
    m1 = PARAMETROS["FURGONETA_CONV"]
    viajes_1 = int(np.ceil(num_paquetes / m1["capacidad"]))
    km_internos = simular_kilometros_tsp(
        barrio_lat_centro, barrio_lon_centro, df_destinos_barrio
    )
    km_totales_1 = (distancia_troncal * 2 * viajes_1) + km_internos
    costo_1 = (km_totales_1 * m1["costo_km"]) + (
        (km_totales_1 / m1["v_media"]) * m1["costo_hora"]
    )
    co2_1 = (km_totales_1 * m1["co2_km"]) / 1000.0

    informe_final["M1: Furgoneta Combustión desde CC"] = {
        "Km Recorridos": km_totales_1,
        "Número Viajes": viajes_1,
        "Emisiones CO2 (kg)": co2_1,
        "Costo Total (€)": costo_1,
    }

    # --- MODELO 2: Furgoneta Eléctrica desde CC ---
    m2 = PARAMETROS["FURGONETA_ELEC"]
    viajes_2 = int(np.ceil(num_paquetes / m2["capacidad"]))
    km_totales_2 = (distancia_troncal * 2 * viajes_2) + km_internos
    costo_2 = (km_totales_2 * m2["costo_km"]) + (
        (km_totales_2 / m2["v_media"]) * m2["costo_hora"]
    )

    informe_final["M2: Furgoneta Eléctrica desde CC"] = {
        "Km Recorridos": km_totales_2,
        "Número Viajes": viajes_2,
        "Emisiones CO2 (kg)": 0.0,
        "Costo Total (€)": costo_2,
    }

    # --- MODELO 3: CC -> Microhub -> Bicicleta ---
    m3 = PARAMETROS["BICICLETA_CARGO"]
    viajes_bike = int(np.ceil(num_paquetes / m3["capacidad"]))
    km_abastecimiento_hub = distancia_troncal * 2
    costo_camion_hub = (
        km_abastecimiento_hub * PARAMETROS["FURGONETA_CONV"]["costo_km"]
    )
    co2_camion_hub = (
        km_abastecimiento_hub * PARAMETROS["FURGONETA_CONV"]["co2_km"]
    ) / 1000.0

    km_bike_internos = km_internos * 1.15  # Desvío estimado por carriles bici
    costo_bike = (km_bike_internos * m3["costo_km"]) + (
        (km_bike_internos / m3["v_media"]) * m3["costo_hora"]
    )

    informe_final["M3: CC -> Microhub -> Bicicleta"] = {
        "Km Recorridos": km_abastecimiento_hub + km_bike_internos,
        "Número Viajes": 1 + viajes_bike,
        "Emisiones CO2 (kg)": co2_camion_hub,
        "Costo Total (€)": costo_camion_hub + costo_bike + m3["fijo_hub_dia"],
    }

    # --- MODELO 4: CC -> PUDO -> Entrega a pie ---
    m4 = PARAMETROS["PUDO_A_PIE"]
    viajes_pie = int(np.ceil(num_paquetes / m4["capacidad"]))
    # Trayectos radiales (ida y vuelta) desde el local PUDO por cada paquete
    km_repartidor_pie = (
        df_destinos_barrio.apply(
            lambda r: calcular_haversine(
                barrio_lat_centro,
                barrio_lon_centro,
                r["Latitude"],
                r["Longitude"],
            ),
            axis=1,
        ).sum()
        * 2
    )
    costo_pie = (km_repartidor_pie / m4["v_media"]) * m4["costo_hora"]

    informe_final["M4: CC -> PUDO -> Entrega a pie"] = {
        "Km Recorridos": km_abastecimiento_hub + km_repartidor_pie,
        "Número Viajes": 1 + viajes_pie,
        "Emisiones CO2 (kg)": co2_camion_hub,
        "Costo Total (€)": costo_camion_hub
        + costo_pie
        + (num_paquetes * m4["comision_pudo"]),
    }

    # --- MODELO 5: CC -> PUDO -> Recogida por el consumidor ---
    m5 = PARAMETROS["PUDO_CONSUMIDOR"]
    km_clientes_radial = km_repartidor_pie
    co2_clientes_motores = (
        km_clientes_radial * m5["co2_km_estimado_cliente"]
    ) / 1000.0

    informe_final["M5: CC -> PUDO -> Recogida Cliente"] = {
        "Km Recorridos": km_abastecimiento_hub + km_clientes_radial,
        "Número Viajes": 1 + num_paquetes,  # 1 del camión + cada cliente individual
        "Emisiones CO2 (kg)": co2_camion_hub + co2_clientes_motores,
        "Costo Total (€)": costo_camion_hub
        + (num_paquetes * m5["comision_pudo"]),
    }

    # =====================================================================
    # 7. IMPRESIÓN ORDENADA DE LA MATRIZ DE RESULTADOS
    # =====================================================================
    df_resultados_colab = pd.DataFrame(informe_final).T
    print(
        f"📊 --- RESULTADOS COMPARATIVOS PARA EL BARRIO: {barrio_activo.upper()} ---"
    )
    print(f"• Centro Logístico de Origen Real: {cc_seleccionado['Location']}")
    print(
        f"• Distancia desde el CC al sector urbano: {distancia_troncal:.2f} km"
    )
    print(f"• Total de paquetes a distribuir: {num_paquetes}\n")

    # Renombrar los índices de las columnas para la presentación formal de la matriz
    df_resultados_colab.columns = [
        "Km Recorridos",
        "Número Viajes",
        "Emissions CO2 (kg)",
        "Costo Total (€)",
    ]
    print(df_resultados_colab.round(2).to_string())